In [1]:
# ============================================================
# 02 - Data Preprocessing
# Project : E-Commerce Customer Churn Analysis
# Author  : Alexander Lau Poung Jie
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import os

os.makedirs("output", exist_ok=True)

# 重新載入原始資料（每個 Notebook 都從原始資料開始，保持獨立性）
df = pd.read_excel("data/E Commerce Dataset.xlsx", sheet_name="E Comm")

print(f"原始資料: {df.shape[0]} 列 x {df.shape[1]} 欄")
print(f"缺失值總數: {df.isnull().sum().sum()}")
print(f"\nChurn 分佈:")
print(df["Churn"].value_counts())

原始資料: 5630 列 x 20 欄
缺失值總數: 1856

Churn 分佈:
Churn
0    4682
1     948
Name: count, dtype: int64


In [2]:
# ============================================================
# Cell 2 - 缺失值填補
# ============================================================
# 策略：用中位數填補數字欄位
# 為什麼用中位數而不是平均數：
#   - 平均數對極端值敏感（如少數超高消費客戶會拉高平均）
#   - 中位數更能代表「典型客戶」的行為
# 為什麼不直接刪除：
#   - 缺失比例只有 4-5%，刪除會損失有效資料
#   - 缺失值分佈不一定與流失相關，刪除可能造成偏誤

# 找出有缺失的數字欄位
numeric_cols_with_na = df.select_dtypes(include=[np.number]).columns[
    df.select_dtypes(include=[np.number]).isnull().any()
].tolist()

print("需要填補的欄位：")
for col in numeric_cols_with_na:
    median_val = df[col].median()
    missing_n  = df[col].isnull().sum()
    df[col].fillna(median_val, inplace=True)
    print(f"  {col:<35} 缺失: {missing_n} → 填補中位數: {median_val:.2f}")

print(f"\n填補後缺失值總數: {df.isnull().sum().sum()}")

需要填補的欄位：
  Tenure                              缺失: 264 → 填補中位數: 9.00
  WarehouseToHome                     缺失: 251 → 填補中位數: 14.00
  HourSpendOnApp                      缺失: 255 → 填補中位數: 3.00
  OrderAmountHikeFromlastYear         缺失: 265 → 填補中位數: 15.00
  CouponUsed                          缺失: 256 → 填補中位數: 1.00
  OrderCount                          缺失: 258 → 填補中位數: 2.00
  DaySinceLastOrder                   缺失: 307 → 填補中位數: 3.00

填補後缺失值總數: 0


/var/folders/gs/y0jdnkc17q97smr9chlnb5kh0000gn/T/ipykernel_90968/1379299632.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)


In [3]:
# ============================================================
# Cell 3 - 類別變數編碼（文字 → 數字）
# ============================================================
# 為什麼需要編碼：
#   機器學習模型只接受數字，文字型別（object）必須先轉換
#
# 方法：One-Hot Encoding
#   把一個類別欄位拆成多個 0/1 欄位
#   例如：Gender（Male/Female）→ Gender_Male（1/0）
#
# 為什麼不用 Label Encoding（直接轉成 0,1,2...）：
#   Label Encoding 會讓模型誤以為類別之間有大小順序
#   例如：Male=0, Female=1 會被誤解為 Female > Male
#   One-Hot Encoding 避免這個問題

# 找出文字欄位
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(f"類別變數: {cat_cols}")
print(f"\n各類別變數的唯一值：")
for col in cat_cols:
    print(f"  {col}: {df[col].unique()}")

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 移除不需要的欄位
df_encoded.drop(columns=["CustomerID"], inplace=True, errors="ignore")

print(f"\n編碼前欄位數: {df.shape[1]}")
print(f"編碼後欄位數: {df_encoded.shape[1]}")
print(f"\n新增的欄位:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)

類別變數: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']

各類別變數的唯一值：
  PreferredLoginDevice: ['Mobile Phone' 'Phone' 'Computer']
  PreferredPaymentMode: ['Debit Card' 'UPI' 'CC' 'Cash on Delivery' 'E wallet' 'COD' 'Credit Card']
  Gender: ['Female' 'Male']
  PreferedOrderCat: ['Laptop & Accessory' 'Mobile' 'Mobile Phone' 'Others' 'Fashion' 'Grocery']
  MaritalStatus: ['Single' 'Divorced' 'Married']

編碼前欄位數: 20
編碼後欄位數: 30

新增的欄位:
['PreferredLoginDevice_Mobile Phone', 'PreferredLoginDevice_Phone', 'PreferredPaymentMode_COD', 'PreferredPaymentMode_Cash on Delivery', 'PreferredPaymentMode_Credit Card', 'PreferredPaymentMode_Debit Card', 'PreferredPaymentMode_E wallet', 'PreferredPaymentMode_UPI', 'Gender_Male', 'PreferedOrderCat_Grocery', 'PreferedOrderCat_Laptop & Accessory', 'PreferedOrderCat_Mobile', 'PreferedOrderCat_Mobile Phone', 'PreferedOrderCat_Others', 'MaritalStatus_Married', 'MaritalStatus_Single']


In [5]:
# ============================================================
# Cell 4 - 標準化 + 處理不平衡資料
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# --- 步驟 1：分離特徵與目標變數 ---
X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"]

print(f"特徵數量: {X.shape[1]}")
print(f"樣本數量: {X.shape[0]}")

# --- 步驟 2：標準化數字欄位 ---
# 為什麼標準化：
#   不同變數尺度差異很大（Tenure: 0-60, CashbackAmount: 0-300+）
#   不標準化的話，數字大的變數會主導模型，產生偏誤
#   標準化後每個變數都是「偏離平均幾個標準差」，公平比較

numeric_features = X.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
X[numeric_features] = scaler.fit_transform(X[numeric_features])

print(f"\n標準化完成")
print(f"Tenure 標準化後 — 平均: {X['Tenure'].mean():.4f}, SD: {X['Tenure'].std():.4f}")

# --- 步驟 3：處理不平衡資料（Oversampling） ---
# 問題：83.2% 留存 vs 16.8% 流失
# 如果不處理，模型會偏向預測「全部留存」
# 就能聲稱 83% 準確率，但完全無法識別流失客戶
#
# 方法：Oversampling（過採樣）
# 把少數類別（流失）重複抽樣，直到與多數類別數量相同
# 優點：簡單有效
# 缺點：可能 overfitting（之後用交叉驗證處理）

X_train = pd.concat([X, y], axis=1)
retained = X_train[X_train["Churn"] == 0]
churned  = X_train[X_train["Churn"] == 1]

churned_upsampled = resample(
    churned,
    replace=True,
    n_samples=len(retained),
    random_state=42
)

df_balanced = pd.concat([retained, churned_upsampled])
X_balanced  = df_balanced.drop(columns=["Churn"])
y_balanced  = df_balanced["Churn"]

print(f"\n處理前 — 留存: {len(retained)}, 流失: {len(churned)}")
print(f"處理後 — 留存: {sum(y_balanced==0)}, 流失: {sum(y_balanced==1)}")
print(f"總樣本數: {len(y_balanced)}")

# --- 步驟 4：儲存處理好的資料 ---
X_balanced.to_csv("data/X_balanced.csv", index=False)
y_balanced.to_csv("data/y_balanced.csv", index=False)

print(f"\n✅ Preprocessing 完成！")
print(f"   儲存至 data/X_balanced.csv")
print(f"   儲存至 data/y_balanced.csv")

特徵數量: 29
樣本數量: 5630

標準化完成
Tenure 標準化後 — 平均: 0.0000, SD: 1.0001

處理前 — 留存: 4682, 流失: 948
處理後 — 留存: 4682, 流失: 4682
總樣本數: 9364

✅ Preprocessing 完成！
   儲存至 data/X_balanced.csv
   儲存至 data/y_balanced.csv
